# Tutorial 3 — Bayesian inversion of the real data of Point034

This notebook applies the inversion of tutorial 2 to measurements made in a real river bed, at the MOLONARI monitoring point Point034 (two weeks of June and July 2016, one measurement every 15 minutes):

1. the processed data of the point are read from `.data/Point034/` and plotted;
2. the direct model is solved with a first guess of the parameters, to see how far it is from the measurements;
3. the MCMC (DREAM, several chains) samples the posterior distribution of the parameters, and the chains are inspected as in tutorial 2;
4. the posterior is read without true values to compare it with: what the data determine, and what they do not;
5. the Darcy velocity and the temperatures given by the posterior are compared with the first guess and with the measurements.

Tutorial 2 explains the settings and the MCMC in more detail and checks them on data whose parameters are known; read it first. The MCMC takes about four minutes.

**The head difference is already in metres.** The processed files of a point give the head difference in metres: the calibration of the pressure sensor, which converts its voltage into a head, was applied when the raw data were processed, and must not be applied again. The data set is a copy of `data/Point034_processed/` of MOLONARI-computo.

**Which parameters are estimated.** As in tutorial 2, only the intrinsic permeability $k$ and the thermal conductivity of the solid $\lambda_s$; the porosity $n$ and the volumetric heat capacity of the solid $\rho_s c_s$ are fixed to the values of the former demonstration notebook of `pyheatmy` on these data, and the lateral source $q_s$ is fixed to zero.


In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyheatmy import Column, Layer
from pyheatmy.config import C_W, LAMBDA_W, RHO_W, ZERO_CELSIUS

from bayesian_inversion import Prior, StateSpace, gelman_rubin, run_mcmc

## 1. Settings

- **Data set.** The point and the depths of the sensors of its shaft. The accuracy of the temperature sensors, `SIGMA_T`, is not known as well as the noise of a synthetic data set: 0.1 K is an assumption, discussed in section 7.
- **First guess.** The parameters of the homogeneous layer of the former demonstration notebook, used in section 3 only: the chains of the MCMC start from points drawn from the priors.
- **Grid, priors, MCMC and seed.** The same as in tutorial 2.


In [ ]:
POINT_NAME = "Point034"
DATA_DIR = Path(".data") / POINT_NAME

DEPTH_SENSORS = [0.1, 0.2, 0.3, 0.4]   # m below the river bed; the last sensor gives the bottom boundary condition
OFFSET = 0.0
NB_CELLS = 100
SIGMA_T = 0.1                          # assumed accuracy of the temperature sensors, K

PRIORS = {
    "IntrinK": ((1e-15, 1e-10), 1e-12),   # (range, step): estimated, m2
    "n": (0.1, 0),                        # (value, 0): fixed
    "lambda_s": ((1.0, 5.0), 0.1),        # estimated, W/m/K
    "rhos_cs": (4e6, 0),                  # fixed, J/m3/K
    "q_s": (0.0, 0),                      # fixed, 1/s
}
ESTIMATED = ["IntrinK", "lambda_s"]
PARAMETERS = ["IntrinK", "n", "lambda_s", "rhos_cs", "q_s"]   # order of pyheatmy's Param
FIRST_GUESS = {"IntrinK": 1e-12, "lambda_s": 2.0}             # the estimated parameters only

NB_CHAIN = 10
NB_ITER = 300
NB_BURN_IN_MAX = 100
DISCARDED_FRACTION = 0.5
OUTLIER_IQR_FACTOR = 2.0   # outlier chains, as in DREAM (Vrugt et al., 2009)

SEED = 20261026

## 2. Reading the data

The files have the format of the synthetic data set of tutorials 1 and 2, and are read in the same way: dates as `yy/mm/dd HH:MM`, temperatures converted from °C to kelvins, and the head difference used as written, in metres. The two files must have the same dates; this is checked here, and `Column` checks that no value is missing in the head difference, in the river temperature and at the deepest sensor.


In [ ]:
DATE_FORMAT = "%y/%m/%d %H:%M"

pressures = pd.read_csv(DATA_DIR / "processed_pressures.csv")
thermal = pd.read_csv(DATA_DIR / "processed_temperatures.csv")
assert (pressures.iloc[:, 0] == thermal.iloc[:, 0]).all(), "the two files must have the same dates"

dates = list(pd.to_datetime(pressures.iloc[:, 0], format=DATE_FORMAT))
head_difference = pressures.iloc[:, 1].to_numpy()
river_temperature = pressures.iloc[:, 2].to_numpy() + ZERO_CELSIUS
sensor_temperatures = thermal.iloc[:, 1:].to_numpy() + ZERO_CELSIUS

dH_measures = list(zip(dates, zip(head_difference, river_temperature)))
T_measures = list(zip(dates, sensor_temperatures))
print(f"{len(dates)} time steps, from {dates[0]} to {dates[-1]}, {sensor_temperatures.shape[1]} temperature sensors")
print(f"Head difference between {head_difference.min():.4f} and {head_difference.max():.4f} m")
print(f"Missing temperatures at the sensors: {int(np.isnan(sensor_temperatures).sum())}")

The head difference is the head in the river minus the head at the bottom of the column. It is negative here: the head is higher below the river bed than in the river, so the water flows upwards, from the aquifer to the river, at a velocity that depends on $k$. It is also small, a few centimetres over the 40 cm of the column. The temperatures in the river bed follow the daily cycle of the river, damped and delayed with depth.


In [ ]:
fig, (ax_head, ax_temperature) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_head.plot(dates, 100 * head_difference)
ax_head.set_ylabel("head difference (cm)")
ax_temperature.plot(dates, river_temperature - ZERO_CELSIUS, color="black", linewidth=0.8, label="river")
for depth, series in zip(DEPTH_SENSORS, sensor_temperatures.T):
    ax_temperature.plot(dates, series - ZERO_CELSIUS, linewidth=0.8, label=f"{depth} m")
ax_temperature.set_ylabel("T (°C)")
ax_temperature.legend(ncol=5)
ax_head.set_title(f"Measurements of {POINT_NAME}")
fig.tight_layout()

## 3. Column, priors and first guess

The column and the `StateSpace` of the priors are built as in tutorial 2. Before the MCMC, the direct model is solved once with `FIRST_GUESS` and the fixed parameters: the root-mean-square error (RMSE) at each inner sensor measures how far this model is from the measurements, and is compared in section 6 with the RMSE of the posterior.


In [ ]:
FIXED = {name: PRIORS[name][0] for name in PARAMETERS if name not in ESTIMATED}


def new_column():
    return Column(
        river_bed=DEPTH_SENSORS[-1],
        depth_sensors=DEPTH_SENSORS,
        offset=OFFSET,
        dH_measures=dH_measures,
        T_measures=T_measures,
        nb_cells=NB_CELLS,
        rac=tempfile.mkdtemp(),
    )


def solve(parameters):
    # Direct model with one set of parameters for the layer; returns the solved column
    solved = new_column()
    solved.set_layers(Layer.from_dict({"name": "river bed", "zLow": DEPTH_SENSORS[-1], **parameters}))
    solved.compute_solve_transi(verbose=False)
    return solved


measured = sensor_temperatures[:, :-1].T   # inner sensors, one row per sensor


def rounded(rmse):
    return {depth: round(float(value), 3) for depth, value in zip(DEPTH_SENSORS[:-1], rmse)}


def rmse_at_sensors(solved):
    simulated = solved.get_temperature_at_sensors()[1:-1]
    return np.sqrt(np.nanmean((simulated - measured) ** 2, axis=1))


def plot_temperatures(solved, label):
    simulated = solved.get_temperature_at_sensors()[1:-1]
    fig, ax = plt.subplots(figsize=(10, 4))
    for depth, sim, meas in zip(DEPTH_SENSORS[:-1], simulated, measured):
        line, = ax.plot(dates, sim - ZERO_CELSIUS, label=f"{depth} m, {label}")
        ax.plot(dates, meas - ZERO_CELSIUS, ".", markersize=2, color=line.get_color(), alpha=0.4)
    ax.set_ylabel("T (°C)")
    ax.set_title("Temperatures at the inner sensors: model (lines) and measurements (dots)")
    ax.legend()
    fig.tight_layout()


column = new_column()
column.set_layers(Layer.from_dict({"name": "river bed", "zLow": DEPTH_SENSORS[-1], **FIRST_GUESS, **FIXED}))
space = StateSpace({"river bed": {name: Prior(*PRIORS[name]) for name in PARAMETERS}})

first_guess = solve({**FIRST_GUESS, **FIXED})
rmse_first_guess = rmse_at_sensors(first_guess)
print("RMSE of the first guess at the inner sensors (K):", rounded(rmse_first_guess))
plot_temperatures(first_guess, "first guess")

## 4. MCMC

The MCMC is run and inspected exactly as in tutorial 2, whose section 4 explains the burn-in, the discarded part of the chains, $\hat R$ and the rule that sets aside an outlier chain. The likelihood is Gaussian, with `sigma2` fixed to `SIGMA_T**2` in the `"no sigma"` mode.


In [ ]:
result = run_mcmc(
    column.simulate,
    column.get_measured_temperatures(),
    space,
    typealgo="no sigma",
    sigma2=SIGMA_T**2,
    nb_iter=NB_ITER,
    nb_chain=NB_CHAIN,
    nitmaxburning=NB_BURN_IN_MAX,
    verbose=False,
    seed=SEED,
)

# The recorded states are stored iteration by iteration, one per chain: the starting point, then NB_ITER iterations
history = result.chains()[:, :, 0, :]   # (NB_ITER + 1, NB_CHAIN, parameters) of the only layer
first_kept = int(DISCARDED_FRACTION * NB_ITER)
energies = result.energies().reshape(NB_ITER + 1, NB_CHAIN)

mean_energy = energies[first_kept:].mean(axis=0)
q1, q3 = np.quantile(mean_energy, [0.25, 0.75])
kept_chains = mean_energy <= q3 + OUTLIER_IQR_FACTOR * (q3 - q1)

# R_hat of the kept iterations of the kept chains, in the sampling space of the priors (log10 k), as in the burn-in
mcmc_chains = np.array([state.params for state in result.states]).reshape(NB_ITER + 1, NB_CHAIN, *space.shape)
is_fixed = np.array([[prior.scale == "fixed" for prior in block] for block in space.priors])
r_hat = gelman_rubin(mcmc_chains[first_kept:, kept_chains], is_fixed)[0]

print("Acceptance rate of each chain:", np.round(result.acceptance, 3))
print("Mean energy of each chain over the kept iterations:", np.round(mean_energy, 1))
print(f"Chains set aside: {np.flatnonzero(~kept_chains).tolist()}")
print("R_hat over the kept iterations:", {name: round(float(r_hat[PARAMETERS.index(name)]), 3) for name in ESTIMATED})

In [ ]:
fig, axes = plt.subplots(len(ESTIMATED), 1, figsize=(10, 5), sharex=True)
for ax, name in zip(axes, ESTIMATED):
    values = history[:, :, PARAMETERS.index(name)]
    shown = np.log10(values) if name == "IntrinK" else values
    ax.plot(shown[:, kept_chains], linewidth=0.8)
    if not kept_chains.all():
        ax.plot(shown[:, ~kept_chains], color="grey", linestyle=":", linewidth=1.2)
    ax.axvline(first_kept, color="black", linestyle="--")
    ax.set_ylabel(r"$\log_{10} k$" if name == "IntrinK" else r"$\lambda_s$")
axes[0].set_title("Chains (one colour per chain, dotted grey: set aside); iterations left of the dashed line are discarded")
axes[-1].set_xlabel("iteration")
fig.tight_layout()

## 5. Posterior

There are no true values to compare with: the table gives the 5 %, 50 % and 95 % quantiles of the posterior, next to the range of the prior. The histograms show the bounds of the prior as dashed lines. Where the posterior reaches a bound of its prior, that side of the posterior is set by the prior, not by the data.


In [ ]:
posterior = history[first_kept:, kept_chains].reshape(-1, len(PARAMETERS))
QUANTILES = [0.05, 0.5, 0.95]
rows = []
for name in ESTIMATED:
    low, median, high = np.quantile(posterior[:, PARAMETERS.index(name)], QUANTILES)
    prior_low, prior_high = PRIORS[name][0]
    rows.append({"parameter": name, "prior from": prior_low, "prior to": prior_high,
                 "5 %": low, "median": median, "95 %": high})
pd.DataFrame(rows).set_index("parameter")

In [ ]:
fig, axes = plt.subplots(1, len(ESTIMATED), figsize=(10, 3.5))
for ax, name in zip(axes, ESTIMATED):
    values = posterior[:, PARAMETERS.index(name)]
    bounds = np.array(PRIORS[name][0])
    if name == "IntrinK":
        values, bounds = np.log10(values), np.log10(bounds)
    ax.hist(values, bins=40, density=True)
    for bound in bounds:
        ax.axvline(bound, color="black", linestyle="--")
    ax.set_xlabel(r"$\log_{10} k$" if name == "IntrinK" else r"$\lambda_s$ (W/m/K)")
axes[0].set_title("Posterior; dashed: bounds of the prior", loc="left")
fig.tight_layout()

## 6. Darcy velocity and temperatures

The Darcy velocity is computed with the direct model for a sample of the posterior states. In `pyheatmy` the depth $z$ points downwards, so a negative velocity is an upward flow.

The thermal Péclet number $Pe = \rho_w c_w |q| L / \lambda_m$ compares the heat carried by the water at the Darcy velocity $q$ with the heat conducted over the length $L$ of the column; $\lambda_m$ is the thermal conductivity of the saturated river bed, which `pyheatmy` computes from $n$, $\lambda_s$ and the conductivity of water. When $Pe$ is much smaller than 1, the heat is transported by conduction and the temperatures depend little on the velocity, hence on $k$. They then depend on the thermal diffusivity of the river bed, $\lambda_m / (\rho c)_m$, where $(\rho c)_m = n \rho_w c_w + (1 - n) \rho_s c_s$ is its volumetric heat capacity; the cell prints both, with the median of $\lambda_s$.


In [ ]:
NB_REPLAYED = 50

rng = np.random.default_rng(SEED)
sample = posterior[rng.choice(len(posterior), size=NB_REPLAYED, replace=False)]
velocities = np.array([solve(dict(zip(PARAMETERS, state))).get_flows_solve().mean() for state in sample])

low, median, high = np.quantile(velocities, QUANTILES)
print(f"Darcy velocity, posterior: median {median:.2e} m/s, 90 % interval [{low:.2e}, {high:.2e}] m/s")

n = FIXED["n"]
lambda_s = np.median(posterior[:, PARAMETERS.index("lambda_s")])
lambda_m = (n * LAMBDA_W**0.5 + (1 - n) * lambda_s**0.5) ** 2   # as in pyheatmy
peclet = RHO_W * C_W * np.abs(velocities) * DEPTH_SENSORS[-1] / lambda_m
print(f"Thermal Péclet number over the column: median {np.median(peclet):.3f}, at most {peclet.max():.3f}")

rho_c_m = n * RHO_W * C_W + (1 - n) * FIXED["rhos_cs"]           # as in pyheatmy
print(f"Thermal conductivity of the river bed: {lambda_m:.2f} W/m/K; thermal diffusivity: {lambda_m / rho_c_m:.2e} m2/s")

The temperatures computed with the median of the posterior are compared with the measurements, and their RMSE with the one of the first guess. The standard deviation of the residuals is compared with `SIGMA_T`.


In [ ]:
median_parameters = dict(zip(PARAMETERS, np.median(posterior, axis=0)))
median_column = solve(median_parameters)
rmse_median = rmse_at_sensors(median_column)
residuals = measured - median_column.get_temperature_at_sensors()[1:-1]
print("RMSE at the inner sensors (K), first guess:", rounded(rmse_first_guess))
print("RMSE at the inner sensors (K), median of the posterior:", rounded(rmse_median))
print(f"Standard deviation of the residuals: {np.nanstd(residuals):.3f} K (SIGMA_T: {SIGMA_T} K)")
plot_temperatures(median_column, "median of the posterior")

## 7. Reading the results

- **Convergence.** As in tutorial 2, the burn-in stops by itself, without a warning, $\hat R$ of the kept iterations is at most 1.2, no chain is set aside, and about one proposal in three is accepted.
- **Fit.** The model with the median of the posterior reproduces the measurements: the RMSE at the inner sensors falls from 0.15–0.19 K with the first guess to 0.03–0.04 K.
- **Thermal conductivity.** $\lambda_s$ is well determined, in a narrow interval around 4.35 W/m/K. Since the heat goes through the river bed by conduction (section 6), the temperatures determine the thermal diffusivity of the river bed, about $10^{-6}$ m²/s, a usual value for a saturated sediment, rather than $\lambda_s$ itself: $\lambda_s$ is estimated for the fixed $n$ and $\rho_s c_s$, and other fixed values would give another $\lambda_s$ for nearly the same diffusivity. With $n = 0.1$, a low porosity for a river bed, the conductivity of the solid has to be high.
- **Permeability.** The posterior of $k$ fills its prior from the lower bound, $10^{-15}$ m², up to about $10^{-13}$ m², and is nearly flat in $\log_{10} k$ there: the data give an upper bound of $k$, about its 95 % quantile, and no lower bound. The head difference is small, the upward flow is slow, and the Péclet number stays below about 0.02, so below about $10^{-13}$ m² the temperatures hardly depend on $k$; above it, the flow would change them more than the measurements allow. The median and the 5 % quantile of $k$ follow the lower bound of the prior, not the data.
- **Darcy velocity.** It follows $k$: the flow is upward, from the aquifer to the river, and its magnitude is at most about $3 \times 10^{-8}$ m/s, about 3 mm per day; its small values come from the prior. For the monitoring point, the result is that the exchange is upward and small, below this bound, over these two weeks.
- **Residuals and `SIGMA_T`.** The standard deviation of the residuals, 0.035 K, is below the assumed accuracy, `SIGMA_T` = 0.1 K: the likelihood is therefore wider than the data would allow, and the intervals are conservative. `run_mcmc` can also estimate `sigma2` (any `typealgo` other than `"no sigma"`), but on these data, with the same settings, its burn-in reached its maximum without converging and few proposals were accepted; it needs longer runs.
- **Model.** The river bed is represented by a single homogeneous layer, with fixed $n$, $\rho_s c_s$ and $q_s$; the results are those of this model.

Things to try: lower the lower bound of the prior of $k$ and see the posterior follow it; set `SIGMA_T` to the standard deviation of the residuals; free $n$ and $\rho_s c_s$ and see which combinations the temperatures constrain; or represent the river bed by two layers, as the former notebook `research/multilayer/demoPyheatmyMultilayer.ipynb` of MOLONARI-computo tried.


## Credits

This tutorial continues `demoPyheatmy.ipynb` of MOLONARI-computo, which ran the direct model and the MCMC of `pyheatmy` on the data of Point034:

| Name | Edition | History |
|---|---|---|
| Yunus Emre Aydogan | MOLONARI 2026 | wrote this tutorial, for pyheatmy 0.12 and bayesian_inversion 0.4 |
